In [25]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

In [9]:
data_path = Path("datasets/undefinenull/million-song-dataset-spotify-lastfm/versions/1")

In [10]:
song_data_path = data_path / 'Music Info.csv'
user_data_path = data_path / 'User Listening History.csv'

In [12]:
df_songs = pd.read_csv(song_data_path)

In [13]:
df_songs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50683 entries, 0 to 50682
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   track_id             50683 non-null  object 
 1   name                 50683 non-null  object 
 2   artist               50683 non-null  object 
 3   spotify_preview_url  50683 non-null  object 
 4   spotify_id           50683 non-null  object 
 5   tags                 49556 non-null  object 
 6   genre                22348 non-null  object 
 7   year                 50683 non-null  int64  
 8   duration_ms          50683 non-null  int64  
 9   danceability         50683 non-null  float64
 10  energy               50683 non-null  float64
 11  key                  50683 non-null  int64  
 12  loudness             50683 non-null  float64
 13  mode                 50683 non-null  int64  
 14  speechiness          50683 non-null  float64
 15  acousticness         50683 non-null 

In [14]:
df_songs.columns

Index(['track_id', 'name', 'artist', 'spotify_preview_url', 'spotify_id',
       'tags', 'genre', 'year', 'duration_ms', 'danceability', 'energy', 'key',
       'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness',
       'liveness', 'valence', 'tempo', 'time_signature'],
      dtype='object')

In [15]:
df_songs.drop_duplicates(subset=['spotify_id','year','duration_ms'],inplace=True)

In [16]:
df_songs.duplicated(subset=['spotify_id','year','duration_ms']).sum()

0

In [17]:
df_songs.reset_index(drop=True,inplace=True)

In [ ]:
# some columns contain totally different values so removing them as no role on calculating similarity
rem_cols = ["spotify_preview_url","name","track_id","spotify_id","genre"]

df_col_filtering = df_songs.drop(columns=rem_cols)

In [ ]:
df_col_filtering.columns 

Index(['artist', 'tags', 'year', 'duration_ms', 'danceability', 'energy',
       'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature'],
      dtype='object')

In [20]:
df_col_filtering.isna().sum()

artist                 0
tags                1126
year                   0
duration_ms            0
danceability           0
energy                 0
key                    0
loudness               0
mode                   0
speechiness            0
acousticness           0
instrumentalness       0
liveness               0
valence                0
tempo                  0
time_signature         0
dtype: int64

In [21]:
df_col_filtering.fillna({"tags" : "no_tags"},inplace=True)

In [22]:
df_col_filtering.isna().sum()

artist              0
tags                0
year                0
duration_ms         0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
dtype: int64

In [23]:
# most used tags
(
    df_songs
    .loc[:,"tags"]
    .str.lower()
    .str.split(',')
    .explode()
    .str.strip()
    .value_counts()
    .loc[lambda ser : ser >= 1000]
)

tags
rock            10681
indie            7284
electronic       6592
alternative      6271
pop              4650
                ...  
ska              1088
gothic_metal     1072
grindcore        1040
french           1018
nu_metal         1006
Name: count, Length: 85, dtype: int64

In [26]:
! pip install category_encoders

In [27]:
from sklearn.preprocessing import MinMaxScaler , StandardScaler , OneHotEncoder
from category_encoders.count import CountEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer

In [30]:
df_col_filtering.head(2)

,artist,tags,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,The Killers,"rock, alternative, indie, alternative_rock, in...",2004,222200,0.355,0.918,1,-4.360,1,0.0746,0.001190,0.0,0.0971,0.240,148.114,4
1,Oasis,"rock, alternative, indie, pop, alternative_roc...",2006,258613,0.409,0.892,2,-4.373,1,0.0336,0.000807,0.0,0.2070,0.651,174.426,4


In [31]:
# columns to transform
frequency_encode_cols = ['year']
ohe_cols = ['artist','time_signature','key']
tfidf_cols = 'tags'
standard_scale_cols = ["duration_ms","loudness","tempo"]
min_max_scale_cols = ["danceability","energy","speechiness","acousticness","instrumentalness","liveness","valence"]

In [33]:
# data transformation pipeline

transformer = ColumnTransformer(transformers=[
    ("frequency_encode",CountEncoder(normalize=True,return_df=True),frequency_encode_cols),
    ("ohe",OneHotEncoder(handle_unknown="ignore"),ohe_cols),
    ("tfidf",TfidfVectorizer(max_features=85),tfidf_cols),
    ("standard_scale",StandardScaler(),standard_scale_cols),
    ("min_max_scale",MinMaxScaler(),min_max_scale_cols)
],remainder='passthrough',n_jobs=-1)

In [34]:

transformer

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('frequency_encode', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name

In [35]:
transformer.fit(df_col_filtering)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('frequency_encode', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",-1
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name

In [ ]:
transformed_df = transformer.transform(df_col_filtering)

In [37]:
transformed_df.shape

(50674, 8431)

In [38]:
from sklearn.metrics.pairwise import cosine_similarity

In [39]:
# fetch a song where artist shakira
df_songs.loc[df_songs["artist"]=="Shakira"]

,track_id,name,artist,spotify_preview_url,spotify_id,tags,genre,year,duration_ms,danceability,...,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
1025,TRLWDVU128F932B093,"Whenever, Wherever",Shakira,https://p.scdn.co/mp3-preview/09ddeb4ae33ee6e8...,07PHBDuUmOeZ7jeKSbAbKi,"rock, pop, female_vocalists, singer_songwriter...",NaN,2012,196826,0.787,...,1,-4.967,0,0.0474,0.29800,0.000005,0.2060,0.860,107.674,4
2068,TRILOWN128F426080F,Underneath Your Clothes,Shakira,https://p.scdn.co/mp3-preview/6c5a56058ce04371...,07qRl4PT2lA6O3KN40McLz,"rock, pop, female_vocalists, singer_songwriter...",Pop,2013,224893,0.707,...,8,-5.293,1,0.0298,0.69100,0.000000,0.1030,0.407,82.784,4
2205,TRXLMFJ12903CC06F7,She Wolf,Shakira,https://p.scdn.co/mp3-preview/4dc802fd3b06fcb5...,075xFXR0JDBwFPVinG1ig5,"electronic, pop, female_vocalists, experimenta...",NaN,2009,187866,0.857,...,7,-6.480,1,0.0428,0.32300,0.003220,0.3140,0.868,121.994,4
2469,TRSQAWU128F92EA20F,La Tortura,Shakira,https://p.scdn.co/mp3-preview/e62ae4649f029729...,0ofDrTTcinCUxm7wqCLPQa,"electronic, pop, female_vocalists, singer_song...",Latin,2011,213106,0.741,...,0,-5.904,1,0.0421,0.02300,0.000788,0.1200,0.834,100.001,4
3373,TRINUNP12903CD84D9,Did It Again,Shakira,https://p.scdn.co/mp3-preview/5477eae2283113ff...,0eMNEdcC5OImvrfn79J9dU,"electronic, pop, female_vocalists, experimenta...",NaN,2009,227333,0.869,...,5,-5.069,0,0.0896,0.50900,0.000000,0.0741,0.599,137.955,4
4318,TROLUWR128F92E5858,Te Dejo Madrid,Shakira,https://p.scdn.co/mp3-preview/bcaccba847a5cf53...,0IESLhxv5iqXvMH5mm3z88,"pop, experimental, singer_songwriter, dance, l...",NaN,2001,187333,0.741,...,11,-5.011,1,0.0379,0.02440,0.000015,0.2830,0.926,131.017,4
4599,TRLNLES128F932DA8E,Fool,Shakira,https://p.scdn.co/mp3-preview/4ce283ae87d032f5...,0MttJjfO3pkTHJgVdXqPcP,"rock, pop, female_vocalists, acoustic, pop_rock",Rap,2001,230333,0.640,...,3,-5.848,1,0.0241,0.04560,0.000000,0.0988,0.420,100.994,4
4617,TRUIPZL12903CA0BFE,Rules,Shakira,https://p.scdn.co/mp3-preview/e5c557dc2b36006a...,1f7TZQzs4UikFQIzeWOtOj,"pop, female_vocalists, singer_songwriter, pop_...",NaN,2001,219106,0.692,...,5,-6.365,0,0.0418,0.03590,0.000002,0.1850,0.775,149.000,4
6046,TRAAKDG128F42A0ECB,Hips Don't Lie,Shakira,https://p.scdn.co/mp3-preview/3859547944f57cfb...,01Yj2MCGpjZs34PRlGgz4K,"pop, female_vocalists, singer_songwriter, danc...",Pop,2001,217453,0.777,...,10,-5.867,0,0.0734,0.28400,0.000000,0.4300,0.760,100.003,4
6818,TRBAHID128F4278EAF,Objection (Tango),Shakira,https://p.scdn.co/mp3-preview/bf65095d5ce58358...,0p9QhtUdbyDAQ6k14hQ2i3,"pop, female_vocalists, singer_songwriter, danc...",Pop,2001,222533,0.603,...,11,-5.282,0,0.0677,0.01470,0.000000,0.0246,0.705,179.344,4


In [40]:
def recommend(song_name, song_data, transformed_data, k=10):

    song_row = song_data.loc[song_data["name"] == song_name]

    if song_row.empty:
        print("Search Another Song")
        return None

    song_idx = song_row.index[0]

    input_vector = transformed_data[song_idx].reshape(1, -1)

    similarity_score = cosine_similarity(
        input_vector,
        transformed_data
    ).ravel()

    # Remove the searched song itself
    top_k_idx = np.argsort(similarity_score)[::-1]
    top_k_idx = top_k_idx[top_k_idx != song_idx][:k]

    recommendations = (
        song_data.iloc[top_k_idx]
        [['name', 'artist', 'spotify_preview_url']]
        .reset_index(drop=True)
    )

    return recommendations

In [42]:
f_df = recommend(
    "Despacito",
    df_songs,
    transformed_df,
    k=10
)

In [46]:
f_df.shape

(10, 3)